# GI Dump to Mod Coverter
[![Static Badge](https://img.shields.io/badge/Jupyter_Notebook-F37726?style=for-the-badge)](https://jupyter.org/)

<br>

Converts the collected files of a mod to the required binary files for the mod for the game GI

<br>

## Contributors

|   |   |
|---|---|
| **[Albert Gold](https://github.com/Alex-Au1)** | [![](https://dcbadge.limes.pink/api/shield/367087171154214914?theme=discord-inverted)](https://discordlookup.com/user/367087171154214914) |


<br>

## Requirements
- Python (Version 3.6 or up)

<br>
<br>

## Installation
Choose how to install AGRemap's API

**Option A**: If you want to install through [Pypi](https://pypi.org/project/AnimeGameRemap/), you run the pip install command below


In [ ]:
%pip install -U AnimeGameRemap

In [ ]:
import AnimeGameRemap as AGR

<br>

**Option B**: Alternatively, you can locally import the API from a specific git branch

In [ ]:
import sys

# Note: Make sure the path correctly points where the AGRemap's API is located
sys.path.insert(1, r"../../../Anime Game Remap (for all users)/api/src/py")

import FixRaidenBoss2 as AGR

<br>
<br>

## Initialization
Run the codeblock below to initialize the necessary tools for the conversion process.

In [ ]:
import os
from enum import Enum
from typing import Dict, List


# VBPart: The .buf files that a GI character splits its vertex data across
class VBPart(Enum):
    Position = "Position"
    Blend = "Blend"
    Texture = "Texcoord"


# StrClassifiers: Classifiers for string text
class StrClassifiers(Enum):
    VBPartClassification = AGR.AhoCorasickBuilder().build(data = {"POSITION": VBPart.Position,
                                                                  "NORMAL": VBPart.Position,
                                                                  "TANGENT": VBPart.Position,
                                                                  "BLEND": VBPart.Blend,
                                                                  "COLOR": VBPart.Texture,
                                                                  "TEXCOORD": VBPart.Texture})


class FileService(AGR.FileService):
    @classmethod
    def readTxt(cls, file: str) -> str:
        with open(file, "r", encoding = AGR.FileEncodings.UTF8.value) as f:
            return f.read()

    @classmethod
    def writeBinary(cls, file: str, data: bytes):
        with open(file, "wb") as f:
            f.write(data)


# ibFromDump(file, fixedFile): Converts a dumped ib.txt file back into its .ib file
def ibFromDump(file: str, fixedFile: str):
    ibFile = AGR.IbFile(b"")
    ibFile.readDumpStr(FileService.readTxt(file))
    FileService.writeBinary(fixedFile, ibFile.data)


# vbFromDump(file, fixedPrefix): Converts a dumped vb.txt file back into the Position/Blend/Texcoord
#   .buf files that a GI character splits its vertex data across
def vbFromDump(file: str, fixedPrefix: str):
    # 'readDumpStr' rebuilds the elements out of the dump's own header, so the layout of the mod
    #   does not need to be known ahead of time
    vbFile = AGR.VbFile(b"", [])
    vbFile.readDumpStr(FileService.readTxt(file))

    # Every element belongs to one of the 3 .buf files, and they are laid out in that order, so
    #   each file is one contiguous byte range of every vertex line
    partRanges: Dict[VBPart, List[int]] = {}
    currentSize = 0

    for element in vbFile.elements:
        partKeyword, part = StrClassifiers.VBPartClassification.value.getMaximal(element.name, errorOnNotFound = False)
        if (part is not None):
            partRange = partRanges.setdefault(part, [currentSize, currentSize])
            partRange[1] = currentSize + element.size

        currentSize += element.size

    data = vbFile.data
    bytesPerLine = vbFile.bytesPerLine

    for part in VBPart:
        if (part not in partRanges):
            continue

        startInd, endInd = partRanges[part]
        partData = b"".join(data[lineInd + startInd: lineInd + endInd] for lineInd in range(0, len(data), bytesPerLine))
        FileService.writeBinary(f"{fixedPrefix}{part.value}.buf", partData)

<br>
<br>

## File Setup
Ensure the file paths are set correctly for the following constants:

- **IBPaths**
- **VBPaths**
- **DDSPaths**

In [9]:
import os
import glob
import re


#####################
# Ensure the pathts set in these constants are correct

IBPaths = {}
VBPaths = {}
DDSPaths = {}

#####################


ModFolders = {
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Amber": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Amber\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\AmberCN": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\AmberCN\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Ayaka": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Ayaka\4_0",
    # r"C:\Users\3dark\Downloads\AyakaSpringBloom\AyakaSpringBloom": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\AyakaSpringBloom\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Barbara": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Barbara\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\BarbaraSummertime": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\BarbaraSummertime\4_0",
    # r"E:\Computer\Downloads\CherryHuTao\CherryHuTao": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\CherryHuTao\5_3",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Diluc": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Diluc\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\DilucFlamme": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\DilucFlamme\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Fischl": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Fischl\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\FischlHighness": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\FischlHighness\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Ganyu": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Ganyu\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\GanyuTwilight": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\GanyuTwilight\4_4",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\HuTao": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\HuTao\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Jean": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Jean\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\JeanCN": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\JeanCN\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\JeanSea": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\JeanSea\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Keqing": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Keqing\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\KeqingOpulent": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\KeqingOpulent\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Kirara": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Kirara\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\KiraraBoots": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\KiraraBoots\4_8",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Klee": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Klee\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\KleeBlossomingStarlight": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\KleeBlossomingStarlight\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Lisa": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Lisa\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\LisaStudent": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\LisaStudent\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Mona": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Mona\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\MonaCN": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\MonaCN\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Nilou": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Nilou\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\NilouBreeze": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\NilouBreeze\4_8",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Ningguang": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Ningguang\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\NingguangOrchid": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\NingguangOrchid\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Rosaria": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Rosaria\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\RosariaCN": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\RosariaCN\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Shenhe": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Shenhe\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\ShenheFrostFlower": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\ShenheFrostFlower\4_4",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Xiangling": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Xiangling\4_0",
    # r"E:\Computer\Downloads\XianglingCheer-corrected\XianglingCheer": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\XianglingCheer\5_3",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Xingqiu": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Xingqiu\4_0",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\XingqiuBamboo": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\XingqiuBamboo\4_4",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Arlecchino": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Arlecchino\4_6",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\RaidenShogun": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\RaidenShogun\4_0",

    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\AyakaSpringbloom": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\AyakaSpringbloom\5_4",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\LisaStudent": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\LisaStudent\5_4",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\NilouBreeze": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\NilouBreeze\5_4",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Arlecchino": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Arlecchino\5_4",

    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Kaeya": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Kaeya\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\KaeyaSailwind": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\KaeyaSailwind\4_0"
}

for srcFolder in ModFolders:
    dstFolder = ModFolders[srcFolder]
    ibPaths = glob.glob(os.path.join(srcFolder, "*-ib*.txt"))
    vbPaths = glob.glob(os.path.join(srcFolder, "*-vb0*.txt"))
    ddsPaths = glob.glob(os.path.join(srcFolder, "*.dds"))

    for ibPath in ibPaths:
        ibFileName = os.path.basename(ibPath)
        ibFileName = re.sub(r"-ib.*", ".ib", ibFileName)
        dstIbPath = os.path.join(dstFolder, ibFileName)
        IBPaths[ibPath] = dstIbPath

    for ddsPath in ddsPaths:
        ddsFileName = os.path.basename(ddsPath)
        dstDDSPath = os.path.join(dstFolder, ddsFileName)
        DDSPaths[ddsPath] = dstDDSPath

    if (not vbPaths):
        continue

    vbPath = vbPaths[0]
    vbFilePrefix = os.path.basename(vbPath)
    vbFilePrefix = re.sub(r"-vb0.*", "", vbFilePrefix)
    vbFilePrefix = re.sub(r"Head|Body|Dress|Extra", "", vbFilePrefix)
    VBPaths[vbPath] = os.path.join(dstFolder, vbFilePrefix)

<br>
<br>

## Run the Converter
The code block below converts the dump files into their corresponding binary formats

In [ ]:
import shutil


for srcPath in IBPaths:
    dstPath = IBPaths[srcPath]
    ibFromDump(srcPath, dstPath)

for srcPath in VBPaths:
    filePrefix = VBPaths[srcPath]
    vbFromDump(srcPath, filePrefix)

for srcPath in DDSPaths:
    dstPath = DDSPaths[srcPath]
    shutil.copy2(srcPath, dstPath)